# Collapsed iOS PCA With Sleep Timing And Stress
PCA analysis of student stress and daily behavior based on College Experience Study Dataset: https://www.kaggle.com/datasets/subigyanepal/college-experience-dataset? The dataset tracks Dartmouth students’ mobile sensing data and ecological momentary assessment (EMA) responses across their college experience, including daily behavioral patterns such as sleep, physical activity, mobility, location visits, phone screen time, and media use, along with self-reported stress. In this analysis, we use only the student who uses ios subset of the data and construct daily behavioral features for each day. We then apply PCA to reduce the dimensionality of these daily behavior variables and examine how the resulting behavioral components are associated with reported stress.



Workflow:
1. Define file paths and settings.
2. Load and filter the iOS stress rows.
3. Collapse raw variables into interpretable daily feature blocks.
4. Concatenate metadata and features into one `analysis_df`, then inspect it.
5. Winsorize extreme values, impute missing values, normalize, then inspect the PCA-ready data.
6. Run PCA.
7. Analyze PC-stress associations.
8. Validate and save outputs.



## 1. Specifying global variables


In [1]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

BASE_DIR = Path("./college-experience-dataset/versions/5") 


SENSING_PATH = BASE_DIR / "Sensing" / "sensing.csv"
STEPS_PATH = BASE_DIR / "Sensing" / "steps.csv"
EMA_PATH = BASE_DIR / "EMA" / "general_ema.csv"
OUTPUT_DIR = Path("./PCA_results")

QUALITY_ACTIVITY_MIN = 8 #ensure coverage of data
QUALITY_LOC_MIN = 8
WINSOR_LOWER_Q = 0.01
WINSOR_UPPER_Q = 0.99
TARGET_CUMULATIVE_VARIANCE = 0.80
MIN_MODEL_PCS = 5
TOP_LOADING_N = 5

PCA_FEATURES = [
    "sleep_hours",
    "sleep_midpoint_hours_after_8pm",
    "steps_total",
    "walking_hours",
    "running_hours",
    "biking_hours",
    "vehicle_hours",
    "still_hours",
    "distance_traveled_km",
    "location_visits",
    "home_hours",
    "own_dorm_hours",
    "social_hours",
    "study_hours",
    "food_hours",
    "other_campus_hours",
    "screen_time_hours",
    "media_playing_hours",
    "media_sessions",
]


## 2. Load And Filter

Load and combine data needed for the PCA

In [2]:
sensing_columns = [
    "uid", "day", "is_ios", "quality_activity", "quality_loc",
    "sleep_duration", "sleep_heathkit_dur", "sleep_start", "sleep_end",
    "act_walking_ep_0", "act_running_ep_0", "act_on_bike_ep_0", "act_in_vehicle_ep_0", "act_still_ep_0",
    "loc_dist_ep_0", "loc_visit_num_ep_0", "loc_home_dur", "loc_self_dorm_dur", "loc_social_dur",
    "loc_other_dorm_dur", "loc_study_dur", "loc_food_dur", "loc_leisure_dur", "loc_workout_dur",
    "loc_health_dur", "loc_worship_dur", "unlock_duration_ep_0", "other_playing_duration_ep_0",
    "other_playing_num_ep_0",
]

sensing = pd.read_csv(SENSING_PATH, usecols=sensing_columns)
ema = pd.read_csv(EMA_PATH, usecols=["uid", "day", "stress"])
steps = pd.read_csv(STEPS_PATH, usecols=["uid", "day", "step_ep_0"])

filtered = sensing.merge(ema, on=["uid", "day"], how="inner")
filtered = filtered[
    (filtered["is_ios"] == 1)
    & filtered["stress"].notna()
    & (filtered["quality_activity"] >= QUALITY_ACTIVITY_MIN)
    & (filtered["quality_loc"] >= QUALITY_LOC_MIN)
].copy().reset_index(drop=True)
filtered["day"] = filtered["day"].astype(int)

print(f"Filtered iOS stress rows: {len(filtered)}")
print(f"Students: {filtered['uid'].nunique()}")
print(f"Date range: {filtered['day'].min()} to {filtered['day'].max()}")
display(filtered["stress"].value_counts().sort_index().rename("stress_count"))
filtered.head()


Filtered iOS stress rows: 22262
Students: 195
Date range: 20170907 to 20220615


stress
1.0    4040
2.0    7548
3.0    6327
4.0    3175
5.0    1172
Name: stress_count, dtype: int64

,uid,is_ios,day,act_in_vehicle_ep_0,act_on_bike_ep_0,act_running_ep_0,act_still_ep_0,act_walking_ep_0,loc_dist_ep_0,loc_food_dur,...,other_playing_duration_ep_0,other_playing_num_ep_0,quality_activity,quality_loc,sleep_duration,sleep_end,sleep_start,unlock_duration_ep_0,sleep_heathkit_dur,stress
0,1ff6d7f34acb354430e7323a35ff7703,1,20170907,0,0,7,72169,14223,5358.167217,NaN,...,2400.0,2.0,17,14,10.00,104,24,4869.000,NaN,1.0
1,1ff6d7f34acb354430e7323a35ff7703,1,20170908,0,0,13,79532,6854,4197.640812,NaN,...,3000.0,3.0,17,24,10.00,94,14,6829.032,8.267,4.0
2,1ff6d7f34acb354430e7323a35ff7703,1,20170911,0,0,5,79730,6664,3846.486372,1.166667,...,4200.0,5.0,16,24,9.25,94,20,6450.807,8.683,4.0
3,1ff6d7f34acb354430e7323a35ff7703,1,20170917,0,0,2,80442,5955,3223.581933,0.000000,...,4199.0,6.0,17,24,8.25,100,34,7289.370,8.150,2.0
4,1ff6d7f34acb354430e7323a35ff7703,1,20170926,0,0,8,81686,4705,3147.060624,1.169444,...,7198.0,4.0,15,24,8.75,98,28,5540.510,8.967,5.0


## 3. Collapse Data Into Feature Blocks


In [3]:
# Match daily steps data to the filtered df
steps_for_rows = filtered[["uid", "day"]].merge(steps, on=["uid", "day"], how="left").reset_index(drop=True)

# Metadata columns identify rows that are not PCA variables.
metadata_block = pd.DataFrame({
    "student_id": filtered["uid"],
    "date": pd.to_datetime(filtered["day"].astype(str), format="%Y%m%d"),
    "day": filtered["day"],
    "stress": filtered["stress"],
    "quality_activity": filtered["quality_activity"],
    "quality_loc": filtered["quality_loc"],
}).reset_index(drop=True)

#Sleep duration: if HealthKit available, use, else use model-derived sleep.
sleep_healthkit = filtered["sleep_heathkit_dur"].astype(float)
sleep_model = filtered["sleep_duration"].astype(float)
sleep_source_healthkit = sleep_healthkit.notna().astype(int)

sleep_start_hours_after_8pm = filtered["sleep_start"].astype(float) / 7.5
sleep_end_hours_after_8pm = filtered["sleep_end"].astype(float) / 7.5

# Combine into a block of features for PCA.
sleep_block = pd.DataFrame({
    "sleep_hours": sleep_healthkit.combine_first(sleep_model),
    "sleep_midpoint_hours_after_8pm": (sleep_start_hours_after_8pm + sleep_end_hours_after_8pm) / 2, #midpoint between sleep start and sleep end, as sleep timing feature
    "sleep_source_healthkit": sleep_source_healthkit,
}).reset_index(drop=True)

# unit handeling
movement_block = pd.DataFrame({
    "steps_total": steps_for_rows["step_ep_0"],
    "walking_hours": filtered["act_walking_ep_0"] / 3600,
    "running_hours": filtered["act_running_ep_0"] / 3600,
    "biking_hours": filtered["act_on_bike_ep_0"] / 3600,
    "vehicle_hours": filtered["act_in_vehicle_ep_0"] / 3600,
    "still_hours": filtered["act_still_ep_0"] / 3600,
}).reset_index(drop=True)

location_block = pd.DataFrame({
    "distance_traveled_km": filtered["loc_dist_ep_0"] / 1000,
    "location_visits": filtered["loc_visit_num_ep_0"],
    "home_hours": filtered["loc_home_dur"],
    "own_dorm_hours": filtered["loc_self_dorm_dur"],
    "social_hours": filtered[["loc_social_dur", "loc_other_dorm_dur"]].sum(axis=1, min_count=1),
    "study_hours": filtered["loc_study_dur"],
    "food_hours": filtered["loc_food_dur"],
    "other_campus_hours": filtered[["loc_leisure_dur", "loc_workout_dur", "loc_health_dur", "loc_worship_dur"]].sum(axis=1, min_count=1),
}).reset_index(drop=True)

phone_media_block = pd.DataFrame({
    "screen_time_hours": filtered["unlock_duration_ep_0"] / 3600,
    "media_playing_hours": filtered["other_playing_duration_ep_0"] / 3600,
    "media_sessions": filtered["other_playing_num_ep_0"],
}).reset_index(drop=True)

print("Feature block shapes:")
for name, block in {
    "metadata_block": metadata_block,
    "sleep_block": sleep_block,
    "movement_block": movement_block,
    "location_block": location_block,
    "phone_media_block": phone_media_block,
}.items():
    print(f"{name}: {block.shape}")


Feature block shapes:
metadata_block: (22262, 6)
sleep_block: (22262, 3)
movement_block: (22262, 6)
location_block: (22262, 8)
phone_media_block: (22262, 3)


## 4. Concatenate Data to DF



In [4]:
analysis_df = pd.concat([metadata_block, sleep_block, movement_block, location_block, phone_media_block],axis=1)
# Only these columns enter PCA. Everything else is metadata/outcome.
feature_cols = PCA_FEATURES.copy()

print(f"analysis_df shape: {analysis_df.shape}")
display(analysis_df.head())


analysis_df shape: (22262, 26)


,student_id,date,day,stress,quality_activity,quality_loc,sleep_hours,sleep_midpoint_hours_after_8pm,sleep_source_healthkit,steps_total,...,location_visits,home_hours,own_dorm_hours,social_hours,study_hours,food_hours,other_campus_hours,screen_time_hours,media_playing_hours,media_sessions
0,1ff6d7f34acb354430e7323a35ff7703,2017-09-07,20170907,1.0,17,14,10.000,8.533333,0,15766.0,...,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.352500,0.666667,2.0
1,1ff6d7f34acb354430e7323a35ff7703,2017-09-08,20170908,4.0,17,24,8.267,7.200000,1,9166.0,...,5.0,NaN,NaN,NaN,NaN,NaN,NaN,1.896953,0.833333,3.0
2,1ff6d7f34acb354430e7323a35ff7703,2017-09-11,20170911,4.0,16,24,8.683,7.600000,1,8368.0,...,5.0,18.334722,18.334722,0.000000,1.499167,1.166667,0.999444,1.791891,1.166667,5.0
3,1ff6d7f34acb354430e7323a35ff7703,2017-09-17,20170917,2.0,17,24,8.150,8.933333,1,7071.0,...,2.0,20.885278,20.885278,0.000000,0.000000,0.000000,0.833611,2.024825,1.166389,6.0
4,1ff6d7f34acb354430e7323a35ff7703,2017-09-26,20170926,5.0,15,24,8.967,8.400000,1,5744.0,...,6.0,15.001667,15.001667,0.665556,5.333056,1.169444,0.000000,1.539031,1.999444,4.0


## 5. Remove outliers and Normalize
We remove outlier by winsorize each feature at the 1st and 99th percentiles and replace missing values using each student's median for that feature. Then center and normalize all feature to same scale using zscore. 



In [5]:
raw_feature_df = analysis_df[feature_cols].copy()

# Winsorize extremes.
winsorized_features = raw_feature_df.copy()
winsor_bounds = []
for col in feature_cols:
    lower = winsorized_features[col].quantile(WINSOR_LOWER_Q)
    upper = winsorized_features[col].quantile(WINSOR_UPPER_Q)
    winsorized_features[col] = winsorized_features[col].clip(lower=lower, upper=upper)
    winsor_bounds.append({"feature": col, "lower": lower, "upper": upper})
winsor_bounds = pd.DataFrame(winsor_bounds)

# Replace missing values.
process_features = winsorized_features.copy()
process_summary = []
student_id = analysis_df["student_id"]
for col in feature_cols:
    missing_before = int(process_features[col].isna().sum())
    student_median = process_features.groupby(student_id)[col].transform("median")
    process_features[col] = process_features[col].fillna(student_median)
    global_median = process_features[col].median()
    process_features[col] = process_features[col].fillna(global_median)
    process_summary.append({
        "feature": col,
        "missing_before": missing_before,
        "missing_before_pct": missing_before / len(process_features),
        "global_median_fallback": global_median,
        "missing_after": int(process_features[col].isna().sum())})
process_summary = pd.DataFrame(process_summary)

# Normalization
student_means = process_features.groupby(student_id).transform("mean")
centered_features = process_features - student_means
centered_global_mean = centered_features.mean(axis=0)
centered_global_std = centered_features.std(axis=0, ddof=0)
pca_matrix = ((centered_features - centered_global_mean) / centered_global_std)[feature_cols]

standardization = pd.DataFrame({"feature": feature_cols,"centered_global_mean": centered_global_mean,"centered_global_std": centered_global_std,})

pca_ready_df = pd.concat(
    [analysis_df[["student_id", "date", "day", "stress", "sleep_source_healthkit"]].reset_index(drop=True), pca_matrix.reset_index(drop=True)],
    axis=1,)

print("Preprocessing checks:")
print("Any missing after processing:", bool(pca_matrix.isna().any().any()))
print("Max absolute normalized feature mean:", pca_matrix.mean().abs().max())
print("Normalized feature std range:", pca_matrix.std(ddof=0).min(), "to", pca_matrix.std(ddof=0).max())

display(process_summary)
display(pca_ready_df.head())


Preprocessing checks:
Any missing after processing: False
Max absolute normalized feature mean: 1.2766916463212653e-17
Normalized feature std range: 0.9999999999999998 to 1.0


,feature,missing_before,missing_before_pct,global_median_fallback,missing_after
0,sleep_hours,0,0.000000,7.333000,0
1,sleep_midpoint_hours_after_8pm,0,0.000000,9.066667,0
2,steps_total,1354,0.060821,5830.000000,0
3,walking_hours,0,0.000000,2.955694,0
4,running_hours,0,0.000000,0.003333,0
5,biking_hours,0,0.000000,0.000000,0
6,vehicle_hours,0,0.000000,0.114861,0
7,still_hours,0,0.000000,20.423611,0
8,distance_traveled_km,0,0.000000,6.453965,0
9,location_visits,0,0.000000,3.000000,0


,student_id,date,day,stress,sleep_source_healthkit,sleep_hours,sleep_midpoint_hours_after_8pm,steps_total,walking_hours,running_hours,...,location_visits,home_hours,own_dorm_hours,social_hours,study_hours,food_hours,other_campus_hours,screen_time_hours,media_playing_hours,media_sessions
0,1ff6d7f34acb354430e7323a35ff7703,2017-09-07,20170907,1.0,0,1.145204,0.302471,2.773315,1.758761,-0.021432,...,0.249979,-0.037296,-0.455389,-0.329500,-0.671668,-0.568441,-0.478728,-0.883483,-0.610642,-1.089286
1,1ff6d7f34acb354430e7323a35ff7703,2017-09-08,20170908,4.0,1,0.439212,-0.530910,1.229894,0.359166,-0.001930,...,0.728095,-0.037296,-0.455389,-0.329500,-0.671668,-0.568441,-0.478728,-0.437290,-0.534479,-0.748668
2,1ff6d7f34acb354430e7323a35ff7703,2017-09-11,20170911,4.0,1,0.608682,-0.280896,1.043281,0.323079,-0.027933,...,0.728095,0.968679,2.273279,-0.329500,-0.211036,0.509908,0.523213,-0.523391,-0.382152,-0.067432
3,1ff6d7f34acb354430e7323a35ff7703,2017-09-17,20170917,2.0,1,0.391548,0.552485,0.739975,0.188419,-0.037684,...,-0.706252,1.306901,2.652866,-0.329500,-0.671668,-0.568441,0.356965,-0.332496,-0.382279,0.273186
4,1ff6d7f34acb354430e7323a35ff7703,2017-09-26,20170926,5.0,1,0.724379,0.219133,0.429654,-0.048993,-0.018182,...,1.206211,0.526691,1.777237,-0.070893,0.966961,0.512475,-0.478728,-0.730616,-0.001590,-0.408050


## 6. PCA


In [6]:
X = pca_matrix.to_numpy(dtype=float)
pca = PCA(n_components=len(feature_cols), svd_solver="full")
scores_array = pca.fit_transform(X)
loadings_array = pca.components_.T

# Stabilize PC signs so fore interpretation, now largest PC loading is positive 
for j in range(loadings_array.shape[1]):
    max_idx = int(np.argmax(np.abs(loadings_array[:, j])))
    if loadings_array[max_idx, j] < 0:
        loadings_array[:, j] *= -1
        scores_array[:, j] *= -1

components = [f"PC{i}" for i in range(1, len(feature_cols) + 1)]
scores = pd.DataFrame(scores_array, columns=components)
loadings = pd.DataFrame(loadings_array, index=feature_cols, columns=components).reset_index(names="feature")

explained_variance = pca.explained_variance_
explained_variance_ratio = pca.explained_variance_ratio_
variance = pd.DataFrame({
    "component": components,
    "explained_variance": explained_variance,
    "explained_variance_ratio": explained_variance_ratio,
    "cumulative_explained_variance_ratio": np.cumsum(explained_variance_ratio),
})

top_loading_rows = []
loading_only = loadings.set_index("feature")
for component in components:
    component_loadings = loading_only[component].sort_values(ascending=False)
    for rank, (feature, loading) in enumerate(component_loadings.head(TOP_LOADING_N).items(), 1):
        top_loading_rows.append({"component": component, "direction": "positive", "rank": rank, "feature": feature, "loading": loading})
    for rank, (feature, loading) in enumerate(component_loadings.tail(TOP_LOADING_N).items(), 1):
        top_loading_rows.append({"component": component, "direction": "negative", "rank": rank, "feature": feature, "loading": loading})
top_loadings = pd.DataFrame(top_loading_rows)

pca_scores_df = pd.concat(
    [analysis_df[["student_id", "date", "day", "stress", "sleep_source_healthkit"]].reset_index(drop=True), scores],
    axis=1,
)

display(variance.head(12))
display(top_loadings[top_loadings["component"].isin(["PC1", "PC2", "PC3"])])


,component,explained_variance,explained_variance_ratio,cumulative_explained_variance_ratio
0,PC1,3.332314,0.175377,0.175377
1,PC2,2.791200,0.146899,0.322276
2,PC3,1.905405,0.100280,0.422556
3,PC4,1.210568,0.063711,0.486267
4,PC5,1.077116,0.056688,0.542955
5,PC6,1.010077,0.053160,0.596114
6,PC7,0.957807,0.050409,0.646523
7,PC8,0.929891,0.048939,0.695462
8,PC9,0.891800,0.046935,0.742397
9,PC10,0.863152,0.045427,0.787824


,component,direction,rank,feature,loading
0,PC1,positive,1,location_visits,0.438713
1,PC1,positive,2,steps_total,0.407591
2,PC1,positive,3,walking_hours,0.370254
3,PC1,positive,4,food_hours,0.291299
4,PC1,positive,5,study_hours,0.254726
5,PC1,negative,1,media_playing_hours,-0.168512
6,PC1,negative,2,media_sessions,-0.188247
7,PC1,negative,3,home_hours,-0.188277
8,PC1,negative,4,screen_time_hours,-0.196941
9,PC1,negative,5,still_hours,-0.269912


## 7. Annalyse Principle Components with Stress



In [7]:
# Correlation check of stress and PCs
stress_rank = analysis_df["stress"].rank(method="average")
spearman = pd.DataFrame([
    {
        "component": component,
        "spearman_r": scores[component].rank(method="average").corr(stress_rank),
        "n": len(scores),
    }
    for component in components
])

# Choose top PCs to reach the target cumulative variance.
reaches_target = variance.index[variance["cumulative_explained_variance_ratio"] >= TARGET_CUMULATIVE_VARIANCE]
n_model_pcs = len(variance) if len(reaches_target) == 0 else int(reaches_target[0]) + 1
n_model_pcs = max(n_model_pcs, MIN_MODEL_PCS)
model_components = [f"PC{i}" for i in range(1, n_model_pcs + 1)]

# Regression with top PCs.
model_df = scores[model_components].copy()
model_df["sleep_source_healthkit"] = analysis_df["sleep_source_healthkit"].astype(float)
y = analysis_df["stress"].astype(float)
student_id = analysis_df["student_id"]

X_demeaned = model_df - model_df.groupby(student_id).transform("mean")
y_demeaned = y - y.groupby(student_id).transform("mean")

X_arr = X_demeaned.to_numpy(dtype=float)
y_arr = y_demeaned.to_numpy(dtype=float)
beta, *_ = np.linalg.lstsq(X_arr, y_arr, rcond=None)
fitted = X_arr @ beta
residuals = y_arr - fitted

rank = int(np.linalg.matrix_rank(X_arr))
n = len(y_arr)
n_students = int(student_id.nunique())
df_resid = max(n - n_students - rank, 1)
rss = float(np.dot(residuals, residuals))
tss = float(np.dot(y_arr, y_arr))
sigma2 = rss / df_resid
cov = sigma2 * np.linalg.pinv(X_arr.T @ X_arr)
standard_errors = np.sqrt(np.diag(cov))
t_stats = beta / standard_errors
p_value_normal_approx = [math.erfc(abs(t) / math.sqrt(2)) for t in t_stats]

stress_model = pd.DataFrame({
    "term": model_df.columns,
    "estimate": beta,
    "std_error": standard_errors,
    "t_stat": t_stats,
    "p_value_normal_approx": p_value_normal_approx,
})

model_summary = {
    "model": "linear student fixed-effects OLS fallback",
    "reason": "statsmodels is not installed; student means are subtracted before OLS",
    "n_rows": n,
    "n_students": n_students,
    "n_terms": len(model_df.columns),
    "rank": rank,
    "df_resid": df_resid,
    "within_r_squared": 1 - rss / tss if tss else np.nan,
    "rss": rss,
    "model_pcs": ",".join(model_components),
}

display(spearman.sort_values("spearman_r", key=lambda s: s.abs(), ascending=False).head(10))
display(stress_model)
model_summary


,component,spearman_r,n
1,PC2,-0.211009,22262
18,PC19,-0.078611,22262
4,PC5,-0.077329,22262
2,PC3,0.072628,22262
17,PC18,-0.067186,22262
5,PC6,-0.065240,22262
8,PC9,0.058644,22262
3,PC4,-0.057736,22262
7,PC8,-0.056811,22262
0,PC1,0.051225,22262


,term,estimate,std_error,t_stat,p_value_normal_approx
0,PC1,0.026323,0.003402,7.737235,1.016018e-14
1,PC2,-0.119895,0.003710,-32.315064,4.297493e-229
2,PC3,0.057415,0.004491,12.785357,1.979505e-37
3,PC4,-0.054882,0.005637,-9.736362,2.109722e-22
4,PC5,-0.061505,0.005973,-10.297257,7.249888e-25
5,PC6,-0.064300,0.006179,-10.406169,2.323869e-25
6,PC7,-0.011276,0.006338,-1.778942,7.524925e-02
7,PC8,-0.047217,0.006438,-7.333755,2.237921e-13
8,PC9,0.042350,0.006562,6.453924,1.089904e-10
9,PC10,0.020521,0.006670,3.076359,2.095457e-03


{'model': 'linear student fixed-effects OLS fallback',
 'reason': 'statsmodels is not installed; student means are subtracted before OLS',
 'n_rows': 22262,
 'n_students': 195,
 'n_terms': 12,
 'rank': 12,
 'df_resid': 22055,
 'within_r_squared': 0.07141354039112235,
 'rss': 18852.730254167505,
 'model_pcs': 'PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11'}

## 8. Save Outputs


In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

analysis_df.to_csv(OUTPUT_DIR / "analysis_df_collapsed.csv", index=False)
raw_feature_df.to_csv(OUTPUT_DIR / "pca_input_features_collapsed_raw.csv", index=False)
winsorized_features.to_csv(OUTPUT_DIR / "pca_input_features_winsorized.csv", index=False)
process_features.to_csv(OUTPUT_DIR / "pca_input_features_imputed.csv", index=False)
pca_matrix.to_csv(OUTPUT_DIR / "pca_input_matrix_standardized.csv", index=False)
pca_ready_df.to_csv(OUTPUT_DIR / "pca_ready_df_standardized_with_metadata.csv", index=False)
pca_scores_df.to_csv(OUTPUT_DIR / "pca_scores.csv", index=False)
loadings.to_csv(OUTPUT_DIR / "pca_loadings.csv", index=False)
variance.to_csv(OUTPUT_DIR / "pca_explained_variance.csv", index=False)
top_loadings.to_csv(OUTPUT_DIR / "pca_top_loadings.csv", index=False)
spearman.to_csv(OUTPUT_DIR / "stress_pc_spearman.csv", index=False)
stress_model.to_csv(OUTPUT_DIR / "stress_pc_user_fixed_effects_ols.csv", index=False)
winsor_bounds.to_csv(OUTPUT_DIR / "preprocessing_winsor_bounds.csv", index=False)
process_summary.to_csv(OUTPUT_DIR / "preprocessing_imputation_summary.csv", index=False)
standardization.to_csv(OUTPUT_DIR / "preprocessing_standardization.csv", index=False)

print(f"Wrote outputs to: {OUTPUT_DIR}")


Wrote outputs to: PCA_results
